# Semana 22 — Comunicación de proyectos IA a clientes no técnicos

**Objetivo:** generar los tres documentos de entrega para la ResearchAgent:
README técnico, guía de uso y resumen ejecutivo.
Además, construir un generador reutilizable que puedas usar en cualquier proyecto futuro.

**Al finalizar tendrás** en `proyectos/research-agent/`:
- `docs/README_tecnico.md` — documentación para IT
- `docs/guia_uso.md` — para el equipo que usa el sistema
- `docs/resumen_ejecutivo.md` — para la dirección
- `docs/prompts/clasificador_v1.md` — plantilla de documentación de prompt
- Generador de resumen ejecutivo con LLM a partir de métricas reales

**Coste estimado:** < $0.02 (un par de llamadas a gpt-4o-mini para la generación de texto).

## Celda 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os, json
from pathlib import Path
from datetime import date
from dotenv import load_dotenv

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
load_dotenv(PROJECT_DIR / ".env")
sys.path.insert(0, str(PROJECT_DIR))

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "Define OPENAI_API_KEY en el .env"

DOCS_DIR    = PROJECT_DIR / "docs"
PROMPTS_DIR = DOCS_DIR / "prompts"
DOCS_DIR.mkdir(exist_ok=True)
PROMPTS_DIR.mkdir(exist_ok=True)

HOY = date.today().strftime("%Y-%m-%d")
print(f"✓ Proyecto: {PROJECT_DIR}")
print(f"✓ Directorio docs: {DOCS_DIR}")
print(f"✓ Fecha: {HOY}")

## Celda 1 — README técnico

Documento para el equipo de IT del cliente: cómo desplegar, mantener y depurar.

In [ ]:
README_TECNICO = f"""# ResearchAgent API — Documentación técnica

**Versión:** 1.0  
**Última actualización:** {HOY}  
**Mantenido por:** [Tu nombre] — [tu@email.com]

---

## Arquitectura

```
Cliente HTTP
    │
    ▼
FastAPI (api.py)
    │  ├── security/guardrails.py   ← validación input/output
    │  ├── security/rate_limiter.py ← control de abuso
    │  └── jobs.py (JobStore)       ← estado en memoria
    │
    ▼ asyncio.create_task
ResearchAgent (agent/agent.py)
    │  ├── planner.py    ← genera el plan de búsqueda
    │  ├── executor.py   ← ejecuta herramientas en bucle ReAct
    │  │     ├── tools.py → buscar_web (DuckDuckGo, sin clave)
    │  │     ├── tools.py → leer_url (BeautifulSoup)
    │  │     └── tools.py → buscar_en_corpus (Chroma RAG)
    │  └── synthesizer.py ← genera el informe final
    │
    ▼
observability/ (Tracer + Logger + Langfuse opcional)
```

## Requisitos del sistema

- Python 3.11+
- 512 MB RAM mínimo (1 GB recomendado)
- Acceso a internet (para DuckDuckGo y OpenAI API)
- Puerto 8000 libre (configurable)

## Variables de entorno

Crea un archivo `.env` en la raíz del proyecto:

```bash
# Obligatorio
OPENAI_API_KEY=sk-...

# Opcional — Langfuse (observabilidad)
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com

# Opcional — configuración
MAX_PASOS_AGENTE=6         # máximo pasos del bucle ReAct
MODELO_LLM=gpt-4o-mini     # modelo a usar
```

## Cómo desplegar

### Local
```bash
pip install -r requirements.txt
uvicorn api:app --host 0.0.0.0 --port 8000
```

### Docker
```bash
docker build -t research-agent .
docker run -p 8000:8000 --env-file .env research-agent
```

### Railway / Render
1. Conecta el repositorio en el panel de Railway o Render
2. Define las variables de entorno en el panel de configuración
3. El despliegue es automático en cada push a `main`

## Cómo ejecutar los tests

```bash
# Tests rápidos (sin llamadas a OpenAI)
pytest test_api.py test_security.py -v

# Tests de integración (requieren OPENAI_API_KEY)
pytest -m lento -v
```

## Cómo monitorizar

### Health check
```bash
curl http://localhost:8000/health
# Respuesta esperada: {{"status": "ok", "jobs_activos": 0}}
```

### Logs
- Los logs estructurados (JSON) se escriben en `logs/research_agent.log`
- Cada llamada al LLM registra: tokens usados, coste estimado, latencia
- Los errores incluyen el `job_id` para correlación

### Métricas en Langfuse (si está configurado)
- Dashboard en `https://cloud.langfuse.com` → proyecto `research-agent`
- Métricas disponibles: coste por traza, latencia p95, tasa de error

## Cómo actualizar los prompts

Los prompts están en `agent/synthesizer.py` (función `generar_informe`) y 
`agent/planner.py` (función `crear_plan`). 
Si usas Langfuse, también puedes versionarlos desde el panel web sin tocar código.

## Troubleshooting

| Error | Causa probable | Solución |
|-------|---------------|----------|
| `AuthenticationError` al arrancar | `OPENAI_API_KEY` mal configurada | Verifica el `.env` y reinicia |
| Job siempre en estado `pending` | El servidor no arrancó con `asyncio` | Usa `uvicorn`, no `python api.py` |
| `RateLimitError` de OpenAI | Demasiadas peticiones concurrentes | Reduce `MAX_PASOS_AGENTE` o añade `time.sleep` entre llamadas |
| Respuestas muy lentas (>30s) | DuckDuckGo throttling | Normal; el agente reintenta automáticamente |
| `JobNotFound 404` al consultar estado | Job expiró (reinicio del servidor) | Los jobs se guardan en memoria; reinicia el job |
"""

(DOCS_DIR / "README_tecnico.md").write_text(README_TECNICO)
print("✓ docs/README_tecnico.md generado")
print(f"  {len(README_TECNICO.splitlines())} líneas")

## Celda 2 — Guía de uso

Para el equipo que va a usar el sistema día a día.

In [ ]:
GUIA_USO = f"""# ResearchAgent — Guía de uso

**Versión:** 1.0  
**Última actualización:** {HOY}

---

## Qué hace este sistema

ResearchAgent busca información en internet y en vuestra base de documentos sobre
cualquier tema que le indiquéis, y genera un informe estructurado en minutos.
Lo que antes requería 30-60 minutos de búsqueda manual ahora tarda entre 1 y 3 minutos.

---

## Cómo lanzar una investigación

### Opción A — Desde el cliente web (si está disponible)
1. Abre la URL del sistema
2. Escribe el tema a investigar
3. Haz clic en **Investigar**
4. Espera 1-3 minutos y descarga el informe

### Opción B — Desde la API
```bash
# Lanza una investigación
curl -X POST http://localhost:8000/research \\
  -H 'Content-Type: application/json' \\
  -d '{{"tema": "Aplicaciones de IA en acuicultura gallega 2025"}}'

# Respuesta: {{"job_id": "a1b2c3d4", "status": "pending", ...}}

# Consulta el estado
curl http://localhost:8000/jobs/a1b2c3d4

# Descarga el informe (cuando status = completed)
curl http://localhost:8000/jobs/a1b2c3d4/report
```

---

## Casos donde funciona bien

✓ Temas con presencia en internet (noticias, informes, publicaciones académicas recientes)  
✓ Preguntas de estado del arte: *"Últimas tecnologías de detección de microplásticos en pesca"*  
✓ Comparativas de tecnologías o proveedores  
✓ Síntesis de normativa europea reciente  
✓ Temas en los que tenéis documentación interna cargada en el corpus  

## Casos donde NO funciona bien

✗ **Información confidencial interna** que no está en el corpus: el agente no accede a vuestros sistemas internos.  
✗ **Datos en tiempo real** (precios de lonja hoy, meteorología ahora): el sistema no tiene acceso a feeds en vivo.  
✗ **Temas muy recientes** (últimas 48 horas): DuckDuckGo puede tardar en indexarlos.  
✗ **Preguntas con respuesta única y exacta** (*"¿Cuántos kilos de merluza se descargaron el martes?"*): para eso necesitáis consultar vuestro sistema de gestión directamente.  

---

## Cómo interpretar el informe

Cada informe incluye:

- **Resumen ejecutivo**: dos o tres párrafos con las ideas principales
- **Hallazgos por área**: desglose por subtemas
- **Fuentes consultadas**: lista de URLs que el agente visitó
- **Limitaciones del informe**: qué no pudo encontrar o verificar

**Importante:** el informe es un punto de partida, no una verdad absoluta.
Verifica siempre los datos clave antes de usarlos en una decisión importante.

---

## Qué hacer si el resultado parece incorrecto

1. **Lee las fuentes**: las URLs al final del informe te permiten verificar cada dato directamente.
2. **Reformula la pregunta**: preguntas más concretas producen informes más precisos. *"Normativa europea pesca sostenible 2024"* es mejor que *"normativa pesca"*.
3. **Añade contexto**: *"...enfocado en el sector conservero gallego"* ayuda al agente a filtrar resultados relevantes.
4. **Reporta el caso**: si el informe contiene un error factual claro, envíalo a [tu@email.com] indicando el job_id. Estos casos sirven para mejorar el sistema.

---

## Límites de uso

- Máximo **10 investigaciones por hora** por usuario
- Longitud mínima del tema: 10 caracteres
- Longitud máxima del tema: 2.000 caracteres
- Si recibes el error `429 Too Many Requests`, espera unos minutos antes de lanzar otra investigación

---

## Contacto y soporte

- **Soporte técnico:** [tu@email.com]
- **Tiempo de respuesta:** 24h en días laborables
- **Para reportar bugs:** incluye siempre el `job_id` del informe afectado
"""

(DOCS_DIR / "guia_uso.md").write_text(GUIA_USO)
print("✓ docs/guia_uso.md generado")
print(f"  {len(GUIA_USO.splitlines())} líneas")

## Celda 3 — Métricas simuladas del proyecto

Antes de generar el resumen ejecutivo, define las métricas reales del proyecto.
En un proyecto real, estas vendrían de los logs de Langfuse o del tracer local.

In [ ]:
# Métricas del proyecto (en un proyecto real vienen de logs/Langfuse)
METRICAS_PROYECTO = {
    "cliente": "ANFACO-CYTMA",
    "sector": "industria conservera pesquera gallega",
    "problema_original": (
        "El equipo de innovación dedicaba 2-3 horas semanales a búsquedas manuales "
        "de informes de tendencias tecnológicas en el sector pesquero y acuícola."
    ),
    "solucion": (
        "Agente de investigación autónomo que genera informes estructurados en 1-3 minutos "
        "buscando en internet y en la base documental interna del cliente."
    ),
    # Métricas cuantitativas
    "informes_generados_mes": 87,
    "tiempo_medio_informe_s": 94,
    "tiempo_manual_estimado_min": 45,
    "coste_por_informe_eur": 0.008,
    "latencia_p95_s": 142,
    "tasa_completados": 0.96,
    # Limitaciones honestas
    "limitaciones": [
        {
            "descripcion": "Temas publicados hace menos de 48 horas pueden no aparecer en los resultados",
            "impacto_porcentaje": 4,
            "mitigacion": "El sistema marca esos informes para revisión manual"
        },
        {
            "descripcion": "No accede a bases de datos de pago (EUMOFA, Globefish)",
            "impacto_porcentaje": 15,
            "mitigacion": "Se puede ampliar añadiendo conectores específicos en fase 2"
        },
    ]
}

# Calcula métricas derivadas
horas_ahorradas_mes = (
    METRICAS_PROYECTO["informes_generados_mes"] *
    METRICAS_PROYECTO["tiempo_manual_estimado_min"] / 60
)
coste_total_mes = (
    METRICAS_PROYECTO["informes_generados_mes"] *
    METRICAS_PROYECTO["coste_por_informe_eur"]
)

print("Métricas del proyecto:")
print(f"  Informes generados/mes: {METRICAS_PROYECTO['informes_generados_mes']}")
print(f"  Tiempo medio/informe:   {METRICAS_PROYECTO['tiempo_medio_informe_s']}s")
print(f"  Tasa de éxito:          {METRICAS_PROYECTO['tasa_completados']*100:.0f}%")
print(f"  Horas ahorradas/mes:    {horas_ahorradas_mes:.0f}h")
print(f"  Coste total/mes:        {coste_total_mes:.2f}€")
print(f"  Coste por hora ahorrada: {coste_total_mes/horas_ahorradas_mes:.3f}€")

## Celda 4 — Resumen ejecutivo generado con LLM

A partir de las métricas estructuradas, el LLM genera el texto ejecutivo.
Tú defines los hechos; el LLM los convierte en lenguaje de negocio.

In [ ]:
def generar_resumen_ejecutivo(metricas: dict) -> str:
    """
    Genera el resumen ejecutivo de un proyecto IA a partir de métricas estructuradas.
    El LLM escribe el texto; los hechos los pones tú.
    """
    limitaciones_txt = "\n".join(
        f"- {lim['descripcion']} (afecta ~{lim['impacto_porcentaje']}% de los casos). "
        f"Mitigación: {lim['mitigacion']}."
        for lim in metricas["limitaciones"]
    )

    prompt = f"""Escribe el resumen ejecutivo de fin de proyecto para un cliente empresarial.
Usa lenguaje directo, sin jerga técnica. Máximo 400 palabras.
Estructura: párrafo de contexto → párrafo de resultados → párrafo de limitaciones → párrafo de próximos pasos.

DATOS DEL PROYECTO:
- Cliente: {metricas['cliente']} ({metricas['sector']})
- Problema original: {metricas['problema_original']}
- Solución implementada: {metricas['solucion']}

MÉTRICAS (úsalas todas, tradúcelas a impacto de negocio):
- {metricas['informes_generados_mes']} informes generados en el primer mes
- Tiempo medio por informe: {metricas['tiempo_medio_informe_s']} segundos
  (vs. {metricas['tiempo_manual_estimado_min']} minutos de búsqueda manual)
- Tasa de éxito: {metricas['tasa_completados']*100:.0f}%
- Coste por informe: {metricas['coste_por_informe_eur']:.3f}€
- Latencia p95: {metricas['latencia_p95_s']}s

LIMITACIONES CONOCIDAS (menciónalas con honestidad usando el marco AIR — reconocer, impacto, resolución):
{limitaciones_txt}

PRÓXIMOS PASOS RECOMENDADOS: sugiere 2-3 mejoras concretas con su impacto esperado."""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=600,
    )
    return resp.choices[0].message.content.strip()


print("Generando resumen ejecutivo con gpt-4o-mini...")
resumen_llm = generar_resumen_ejecutivo(METRICAS_PROYECTO)

# Encabezado del documento
RESUMEN_EJECUTIVO = f"""# ResearchAgent — Resumen ejecutivo

**Cliente:** {METRICAS_PROYECTO['cliente']}  
**Fecha:** {HOY}  
**Preparado por:** [Tu nombre]

---

{resumen_llm}

---

## Métricas de referencia

| Métrica | Valor |
|---------|-------|
| Informes generados (primer mes) | {METRICAS_PROYECTO['informes_generados_mes']} |
| Tiempo medio por informe | {METRICAS_PROYECTO['tiempo_medio_informe_s']}s |
| Tiempo manual equivalente | {METRICAS_PROYECTO['tiempo_manual_estimado_min']} min |
| Tasa de éxito | {METRICAS_PROYECTO['tasa_completados']*100:.0f}% |
| Coste por informe | {METRICAS_PROYECTO['coste_por_informe_eur']:.3f}€ |
| Latencia p95 | {METRICAS_PROYECTO['latencia_p95_s']}s |
"""

(DOCS_DIR / "resumen_ejecutivo.md").write_text(RESUMEN_EJECUTIVO)
print("✓ docs/resumen_ejecutivo.md generado")
print()
print("--- Contenido generado ---")
print(resumen_llm)

## Celda 5 — Documentación de prompts

In [ ]:
# Lee el prompt actual del synthesizer para documentarlo
SYNTHESIZER_PATH = PROJECT_DIR / "agent" / "synthesizer.py"

# Extracto del prompt de síntesis (hardcoded como fallback en synthesizer.py)
PROMPT_ACTUAL = """Eres un investigador experto. Con base en la siguiente investigación,
genera un informe detallado y bien estructurado en español.

Tema: {tema}

Evidencias recopiladas:
{evidencias}

El informe debe incluir:
1. Resumen ejecutivo (2-3 párrafos)
2. Hallazgos principales organizados por área temática
3. Conclusiones y recomendaciones
4. Limitaciones de la investigación"""

DOC_PROMPT = f"""# Prompt: Sintetizador de informes de investigación

**Nombre:** synthesizer_informe  
**Versión:** 1.0  
**Última actualización:** {HOY}  
**Modelo:** gpt-4o-mini  
**Temperatura:** 0.3  

---

## Propósito

Sintetiza las evidencias recopiladas por el agente (búsquedas web, URLs leídas,
corpus interno) en un informe estructurado en español.

## Variables del prompt

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `{{tema}}` | string | El tema de investigación solicitado por el usuario |
| `{{evidencias}}` | string | JSON con las evidencias recopiladas por el ejecutor |

## Cuándo modificarlo

- Si el cliente pide informes en otro idioma: cambia "en español" e indica el idioma
- Si necesitas una sección adicional: añádela a la lista numerada del final
- Si los informes son demasiado largos: añade `Máximo N palabras por sección`
- Si el cliente tiene un formato corporativo concreto: añade un ejemplo en la sección de instrucciones

## Prompt actual

```
{PROMPT_ACTUAL}
```

## Métricas de rendimiento

| Métrica | Valor |
|---------|-------|
| Tokens input (media) | ~1.200 |
| Tokens output (media) | ~500 |
| Coste por llamada | ~$0.0003 |
| Latencia media | ~4s |

## Historial de cambios

- **v1.0** ({HOY}): Versión inicial. Estructura de 4 secciones.
"""

(PROMPTS_DIR / "synthesizer_v1.md").write_text(DOC_PROMPT)
print("✓ docs/prompts/synthesizer_v1.md generado")

# Lista todos los documentos generados
print()
print("Documentos de entrega generados:")
for f in sorted(DOCS_DIR.rglob("*.md")):
    rel = f.relative_to(PROJECT_DIR)
    lineas = len(f.read_text().splitlines())
    print(f"  {rel}  ({lineas} líneas)")

## Celda 6 — Generador de demo con datos reales del cliente

Antes de una reunión con el cliente, prepara tres casos de demostración:
uno bueno, uno límite y uno de fallo conocido.

In [ ]:
CASOS_DEMO = [
    {
        "tipo": "caso_bueno",
        "etiqueta": "✓ Caso estándar (muestra primero)",
        "tema": "Aplicaciones de visión artificial en líneas de procesado de conservas de pescado",
        "por_que": "Tema con muchas publicaciones recientes, bien indexado, resultado predecible",
    },
    {
        "tipo": "caso_limite",
        "etiqueta": "⚠ Caso límite (muestra honestidad)",
        "tema": "Normativa gallega de vertidos en puertos pesqueros actualización 2025",
        "por_que": "Normativa regional reciente, puede no estar bien indexada. El agente encontrará algo pero puede ser incompleto.",
    },
    {
        "tipo": "caso_fallo",
        "etiqueta": "✗ Caso de fallo conocido (demuestra transparencia)",
        "tema": "Precio de la anchoa en lonja de Burela hoy",
        "por_que": "Datos en tiempo real — el sistema no los tiene. Muéstralo para demostrar que conoces los límites del sistema.",
    },
]

print("Plan de demostración:")
print("=" * 60)
for i, caso in enumerate(CASOS_DEMO, 1):
    print(f"\nCaso {i}: {caso['etiqueta']}")
    print(f"  Tema:    {caso['tema']}")
    print(f"  Por qué: {caso['por_que']}")

print()
print("Estructura de la demo (20 min):")
estructura = [
    ("00:00", "02:00", "El problema que resolvemos (en palabras del cliente)"),
    ("02:00", "08:00", "Demo Caso 1 — caso estándar con sus datos"),
    ("08:00", "12:00", "Demo Caso 2 — caso límite + cómo lo gestiona"),
    ("12:00", "14:00", "Demo Caso 3 — fallo conocido + transparencia"),
    ("14:00", "17:00", "Métricas: horas ahorradas, coste, tasa de éxito"),
    ("17:00", "20:00", "Preguntas"),
]
for inicio, fin, desc in estructura:
    print(f"  {inicio}–{fin}  {desc}")

## Celda 7 — Traductor de métricas técnicas a lenguaje de negocio

In [ ]:
def traducir_metrica(metrica_tecnica: str, valor, contexto: str = "") -> str:
    """
    Traduce una métrica técnica a lenguaje de negocio usando el LLM.
    Reutilizable para cualquier proyecto.
    """
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Traduce esta métrica técnica a una frase en lenguaje de negocio.
Una sola frase, máximo 25 palabras. Sin jerga técnica.
Contexto del proyecto: {contexto}
Métrica: {metrica_tecnica} = {valor}"""
        }],
        temperature=0.3,
        max_tokens=60,
    )
    return resp.choices[0].message.content.strip()


METRICAS_A_TRADUCIR = [
    ("accuracy", "94%"),
    ("latencia p95", "142 segundos"),
    ("coste por informe", "0.008€"),
    ("tasa de jobs completados", "96%"),
    ("tiempo medio de generación", "94 segundos"),
]

CONTEXTO = "sistema de generación automática de informes de investigación para el sector pesquero gallego"

print("Traducción de métricas técnicas:")
print("=" * 60)
for metrica, valor in METRICAS_A_TRADUCIR:
    traduccion = traducir_metrica(metrica, valor, CONTEXTO)
    print(f"  Técnico:  {metrica} = {valor}")
    print(f"  Negocio:  {traduccion}")
    print()

## Reto — Email de cierre de proyecto

Escribe una función `generar_email_cierre(metricas)` que genere el email que
envías al cliente cuando entregas el proyecto. El email debe:

1. Tener asunto atractivo que no sea genérico (*"Entrega del proyecto"* no vale)
2. Resumir el impacto en **dos frases** sin métricas técnicas
3. Listar los tres entregables con un enlace placeholder para cada uno
4. Incluir una llamada a la acción clara para la reunión de presentación
5. Terminar con una frase que abra la puerta a la siguiente fase

```python
def generar_email_cierre(metricas: dict) -> str:
    # TODO: llama a gpt-4o-mini con un prompt que incluya todos los puntos
    # Tono: profesional pero cercano, sin anglicismos innecesarios
    pass

email = generar_email_cierre(METRICAS_PROYECTO)
print(email)
```

Prueba el resultado con tus propios datos reales del proyecto.

## Resumen de la semana 22

| Entregable | Audiencia | Qué responde |
|-----------|-----------|-------------|
| `docs/README_tecnico.md` | Equipo IT | ¿Cómo despliego y mantengo esto? |
| `docs/guia_uso.md` | Usuarios finales | ¿Cómo saco partido sin romperlo? |
| `docs/resumen_ejecutivo.md` | Dirección | ¿Valió la pena la inversión? |
| `docs/prompts/synthesizer_v1.md` | Cualquiera | ¿Cómo funciona y cómo actualizo los prompts? |
| `generar_resumen_ejecutivo()` | Reutilizable | Genera texto ejecutivo a partir de métricas |
| `traducir_metrica()` | Reutilizable | Convierte métricas técnicas en lenguaje de negocio |

La semana 23 cubre **pricing avanzado y gestión de proyectos IA**: cómo estimar esfuerzo, 
gestionar cambios de alcance y protegerte contractualmente.